# Character Extraction Visualiser

Inspect the output of `character_extraction.py`: per-word orientations,
stroke orientations, refined character bounding boxes, and embedded 40×32 character images.

**Views available:**
1. Full-page overlay — character bounding boxes drawn on the page.
2. Zoom into a region.
3. Single-word detail — word crop with char bboxes + embedded char images.
4. Embedded character images grid.
5. Words table — orientations, stroke angles, and char counts.
6. Browse pages.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────
# Paths follow the same layout produced by character_extraction.py.

IMAGE_ROOT  = r"../data/corpus-1/imgs"
JSON_ROOT   = r"../data/corpus-1/charnet"

DOCUMENT    = "BNE_1001_615_T-55281-18"
PAGE        = "page_10"

In [ ]:
import json, os, sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import numpy as np

PALETTE = list(mcolors.TABLEAU_COLORS.values())  # 10 distinct colours

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 8,
    "axes.titlesize": 11,
})

In [ ]:
# ── Load all data for the selected page ──────────────────────────────

img_path  = Path(IMAGE_ROOT) / DOCUMENT / f"{PAGE}.png"
json_path = Path(JSON_ROOT)  / DOCUMENT / f"{PAGE}.json"
npz_path  = Path(JSON_ROOT)  / DOCUMENT / f"{PAGE}_data.npz"

for p, label in [(img_path, "Image"), (json_path, "JSON"), (npz_path, ".npz")]:
    assert p.exists(), f"{label} not found: {p}"

img_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
img_rgb  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

with open(json_path) as f:
    words = json.load(f)

data = np.load(str(npz_path), allow_pickle=True)

word_orientations        = data["word_orientations"]
word_stroke_orientations = data["word_stroke_orientations"]
page_height              = int(data["page_height"][0])
char_imgs                = data["char_imgs"]        # (N, 40, 32) float32
char_labels              = data["char_labels"]       # (N,) U1
char_word_idx            = data["char_word_idx"]     # (N,) int32

n_words = len(words)
print(f"Image      : {img_rgb.shape[1]}×{img_rgb.shape[0]}")
print(f"Words      : {n_words}")
print(f"Char images: {len(char_imgs)}")
print(f"Page height: {page_height}")

In [ ]:
# ── Helpers: rebuild per-word char tblrs ─────────────────────────────

def load_word_chars(data, n_words):
    """Read char_tblrs_i arrays from the .npz into a list."""
    char_tblrs_list = []
    for i in range(n_words):
        key_t = f"char_tblrs_{i}"
        ct = data[key_t] if key_t in data else np.zeros((0, 4), dtype=np.int32)
        char_tblrs_list.append(ct)
    return char_tblrs_list

char_tblrs_list = load_word_chars(data, n_words)

# Per-word char counts from flat char_word_idx
word_char_counts = np.bincount(char_word_idx, minlength=n_words) if len(char_word_idx) > 0 else np.zeros(n_words, dtype=int)

n_segmented = sum(1 for c in char_tblrs_list if len(c) > 0)
print(f"Words with segmented chars: {n_segmented}/{n_words}")
print(f"Total embedded char images: {len(char_imgs)}")

---
## 1. Full-page overlay

Refined character bounding boxes drawn on the page image using
alternating colours (one colour per word, cycling through a palette).

In [ ]:
# ── Full-page bbox overlay ────────────────────────────────────────────

def draw_char_bboxes(ax, char_tblrs_list, palette=PALETTE, linewidth=1.0):
    """Draw character bounding boxes as coloured rectangles."""
    for wi, tblrs in enumerate(char_tblrs_list):
        if len(tblrs) == 0:
            continue
        for ci, tblr in enumerate(tblrs):
            t, b, l, r = tblr.astype(int)
            colour = palette[(ci + wi) % len(palette)]
            rect = Rectangle((l, t), r - l, b - t,
                              linewidth=linewidth, edgecolor=colour,
                              facecolor="none")
            ax.add_patch(rect)

fig, ax = plt.subplots(1, 1, figsize=(24, 24))
ax.imshow(img_rgb)
draw_char_bboxes(ax, char_tblrs_list)
ax.set_axis_off()
ax.set_title(f"{DOCUMENT} / {PAGE}  —  {len(char_imgs)} character bboxes",
             fontweight="bold")
fig.tight_layout()
plt.show()

---
## 2. Zoom into a region

Set pixel coordinates to zoom into a specific area.

In [ ]:
# ── Crop coordinates (adjust as needed) ──────────────────────────────
X_MIN, X_MAX = 0, 1500
Y_MIN, Y_MAX = 700, 1200

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 8))
ax.imshow(img_rgb)
draw_char_bboxes(ax, char_tblrs_list, linewidth=1.5)
ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(Y_MAX, Y_MIN)
ax.set_axis_off()
ax.set_title(f"Zoom: x=[{X_MIN},{X_MAX}], y=[{Y_MIN},{Y_MAX}]", fontweight="bold")
fig.tight_layout()
plt.show()

---
## 3. Single-word detail

Pick a word index to see its crop with character bounding boxes,
metadata (orientation, stroke angle), and the embedded 40×32 char images.

In [ ]:
# ── Select word ──────────────────────────────────────────────────────
WORD_INDEX = 100   # change this to inspect a different word

In [ ]:
# ── Word crop with char bboxes + embedded images ─────────────────────

wi = WORD_INDEX
word = words[wi]
wt, wb, wl, wr = word["tblr"]
text = word.get("text", "")
tblrs = char_tblrs_list[wi]

# Get embedded char images for this word
word_mask = char_word_idx == wi
word_char_imgs = char_imgs[word_mask]
word_char_labels = char_labels[word_mask]
n_chars = len(word_char_imgs)

# Compute the crop region
word_h = wb - wt
pad = max(1, word_h // 5)
crop_t, crop_b = wt - pad, wb + pad
crop_l, crop_r = wl - pad, wr + pad
if len(tblrs) > 0:
    crop_t = min(crop_t, int(tblrs[:, 0].min()))
    crop_b = max(crop_b, int(tblrs[:, 1].max()))
    crop_l = min(crop_l, int(tblrs[:, 2].min()))
    crop_r = max(crop_r, int(tblrs[:, 3].max()))
crop_t = max(0, crop_t)
crop_b = min(img_rgb.shape[0], crop_b)
crop_l = max(0, crop_l)
crop_r = min(img_rgb.shape[1], crop_r)

word_crop = img_rgb[crop_t:crop_b, crop_l:crop_r].copy()

# -- Plot: word crop + bboxes (top), embedded char images (bottom) --
n_cols = max(n_chars, 1)
fig = plt.figure(figsize=(max(16, n_cols * 1.5), 6))

# Top: word crop with bboxes
ax_crop = fig.add_axes([0.02, 0.4, 0.96, 0.55])
ax_crop.imshow(word_crop)
if len(tblrs) > 0:
    for ci, tblr in enumerate(tblrs):
        t, b, l, r = tblr.astype(int)
        colour = PALETTE[ci % len(PALETTE)]
        rect = Rectangle((l - crop_l, t - crop_t), r - l, b - t,
                          linewidth=1.5, edgecolor=colour, facecolor="none")
        ax_crop.add_patch(rect)
ax_crop.set_axis_off()

ori = word_orientations[wi]
stroke = word_stroke_orientations[wi]
ax_crop.set_title(
    f"Word {wi}: \"{text}\"  |  "
    f"orientation={ori:.2f}°  stroke={stroke:.2f}°  |  "
    f"{n_chars} chars",
    fontsize=11, fontweight="bold",
)

# Bottom: embedded char images row
if n_chars > 0:
    for ci in range(n_chars):
        ax_ch = fig.add_axes([
            0.02 + ci * (0.96 / n_cols),
            0.02,
            0.96 / n_cols - 0.005,
            0.32,
        ])
        ax_ch.imshow(word_char_imgs[ci], cmap="gray", vmin=0, vmax=1)
        lbl = word_char_labels[ci] if ci < len(word_char_labels) else "?"
        ax_ch.set_title(f"'{lbl}'", fontsize=9)
        ax_ch.set_axis_off()

plt.show()

---
## 4. Embedded character images grid

Display a grid of the embedded 40×32 character images with their OCR labels.

In [ ]:
# ── Character image grid ──────────────────────────────────────────────
MAX_SHOW = 200  # max chars to display in the grid
n_show = min(len(char_imgs), MAX_SHOW)
n_grid_cols = 20
n_grid_rows = int(np.ceil(n_show / n_grid_cols))

fig, axes = plt.subplots(n_grid_rows, n_grid_cols,
                         figsize=(n_grid_cols * 0.8, n_grid_rows * 1.0))
axes = np.atleast_2d(axes)
for idx in range(n_grid_rows * n_grid_cols):
    r, c = divmod(idx, n_grid_cols)
    ax = axes[r, c]
    if idx < n_show:
        ax.imshow(char_imgs[idx], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"'{char_labels[idx]}'", fontsize=6, pad=1)
    ax.set_axis_off()

fig.suptitle(f"Embedded char images (showing {n_show}/{len(char_imgs)})",
             fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

---
## 5. Words table

Summary of every word with orientation, stroke angle, and char count.

In [ ]:
import pandas as pd

rows = []
for i, w in enumerate(words):
    n_chars_i = int(word_char_counts[i])
    rows.append({
        "#": i,
        "text": w.get("text", ""),
        "text_score": round(w.get("text_score", 0), 3),
        "n_chars_ocr": len(w.get("chars", [])),
        "n_chars_seg": n_chars_i,
        "orientation": round(float(word_orientations[i]), 2),
        "stroke": round(float(word_stroke_orientations[i]), 2),
    })

df = pd.DataFrame(rows)

styled = (
    df.style
    .background_gradient(subset=["text_score"], cmap="RdYlGn", vmin=0.5, vmax=1.0)
    .background_gradient(subset=["orientation"], cmap="coolwarm")
    .format({
        "text_score": "{:.3f}",
        "orientation": "{:.2f}",
        "stroke": "{:.2f}",
    })
)
styled

---
## 6. Browse pages

List every page that has all output files (image, JSON, .npz).
Change `PAGE` in the configuration cell and re-run.

In [ ]:
img_dir  = Path(IMAGE_ROOT) / DOCUMENT
json_dir = Path(JSON_ROOT)  / DOCUMENT

available = sorted(
    p.stem for p in img_dir.glob("*.png")
    if (json_dir / f"{p.stem}.json").exists()
       and (json_dir / f"{p.stem}_data.npz").exists()
)
print(f"{len(available)} pages with extraction results in '{DOCUMENT}':")
for name in available:
    npz = np.load(str(json_dir / f"{name}_data.npz"), allow_pickle=True)
    n_w = len(json.load(open(json_dir / f"{name}.json")))
    n_c = len(npz["char_imgs"]) if "char_imgs" in npz else 0
    print(f"  {name:20s}  ({n_w:3d} words, {n_c:4d} chars)")